In [ ]:
import pandas as pd

Installing ydata-profiling for data exploration

In [ ]:
!pip install ydata-profiling
import pandas as pd
from ydata_profiling import ProfileReport

# Ensure 'CO2 emission by countries.csv' is uploaded to your Colab environment or correctly mounted from Google Drive.
# If you are reading from Google Drive, ensure it's mounted, and the path is correct.
# from google.colab import files
# uploaded = files.upload()
# Then verify the file name if you used the upload function.

df = pd.read_csv("CO2 emission by countries.csv", encoding="latin1")

profile_raw = ProfileReport(
    df,
    title="CO2 emission by countries Raw Data Report",
    explorative=True,
    correlations={"pearson": {"calculate": True}},
)

profile_raw.to_file("CO2_emission_by_countries_raw_report.html")

Profiling

In [ ]:
# Initial Data Profiling Report

In [ ]:
df[["Country", "Year", "CO2 emission (Tons)", "Population(2022)"]]

Previewing Raw Data

In [ ]:
df.head(100)

In [ ]:
cleanedco2 = df.loc[
    df["CO2 emission (Tons)"] > 0,
    ["Country", "Year", "CO2 emission (Tons)", "Population(2022)"]
]

Filtering data by country, year, population, and CO2 emission

In [ ]:
cleanedco2

In [ ]:
profile = ProfileReport(
    cleanedco2,
    title="Cleaned CO2 Dataset Profiling Report",
    explorative=True
)

profile.to_file("cleanedco2_profile.html")

Generating Profile Report for Cleaned Data

In [ ]:
from google.colab import files

files.download("cleanedco2_profile.html")

The missing values in the population column are a lot

In [ ]:
df[df["Population(2022)"].isna()]

Filter by CO2 emission that is non zero, country and year

In [ ]:
cleanedco2 = df.loc[
    df["CO2 emission (Tons)"] > 0,
    ["Country", "Year", "CO2 emission (Tons)"]
]

Any missing value?

In [ ]:
cleanedco2.head()

In [ ]:
cleanedco2.isna().sum()

Starting Data Profiling

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    cleanedco2,
    title="CO2 Emissions Profiling Report",
    explorative=True
)

profile.to_file("cleanedco2_profile.html")

Profiling View

In [ ]:
from google.colab import files

files.download("cleanedco2_profile.html")

Descriptive analysis

In [ ]:
cleanedco2.describe()

Top emitters

In [ ]:
cleanedco2.groupby("Country")["CO2 emission (Tons)"] \
          .sum() \
          .sort_values(ascending=False)

Bar Chart of Top Emitters

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="darkgrid")
plt.style.use("dark_background")

top_emitters = (
    cleanedco2.groupby("Country")["CO2 emission (Tons)"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(10,5))
sns.barplot(
    data=top_emitters,
    x="CO2 emission (Tons)",
    y="Country",
    order=top_emitters.sort_values("CO2 emission (Tons)")["Country"]
)

plt.title("Top 10 CO2 Emitters")
plt.show()

Yearly trend

In [ ]:
cleanedco2.groupby("Year")["CO2 emission (Tons)"] \
          .sum()

Line chart

In [ ]:
yearly = (
    cleanedco2.groupby("Year")["CO2 emission (Tons)"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(10,5))
sns.lineplot(
    data=yearly,
    x="Year",
    y="CO2 emission (Tons)",
    marker="o"
)

plt.title("Yearly CO2 Emissions Trend")
plt.show()

United States CO2 Emissions Over Time

In [ ]:
us = cleanedco2[cleanedco2["Country"] == "United States"]

us_yearly = (
    us.groupby("Year")["CO2 emission (Tons)"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(10,5))
sns.lineplot(
    data=us_yearly,
    x="Year",
    y="CO2 emission (Tons)",
    marker="o"
)

plt.title("United States CO2 Emissions Over Time")
plt.show()

US Ranking each year

In [ ]:
cleanedco2["rank"] = (
    cleanedco2.groupby("Year")["CO2 emission (Tons)"]
    .rank(ascending=False)
)

us_rank = cleanedco2[cleanedco2["Country"] == "United States"]

us_rank[["Year", "CO2 emission (Tons)", "rank"]]

Overall CO2 Emission Ranking

Outlier Detection

In [ ]:
Q1 = cleanedco2["CO2 emission (Tons)"].quantile(0.25)
Q3 = cleanedco2["CO2 emission (Tons)"].quantile(0.75)

IQR = Q3 - Q1

outliers = cleanedco2[
    (cleanedco2["CO2 emission (Tons)"] < Q1 - 1.5*IQR) |
    (cleanedco2["CO2 emission (Tons)"] > Q3 + 1.5*IQR)
]

Detected Outliers

In [ ]:
outliers

The Outlier Shape

In [ ]:
outliers.shape

Cheking which country dominates outliers

In [ ]:
Dominating_outlier_countries = outliers["Country"].value_counts()

In [ ]:
Dominating_outlier_countries.head()

Checking which year generates outliers

In [ ]:
outliers["Year"].value_counts().sort_index()

I want to confirm if this outliers are expected

In [ ]:
cleanedco2.sort_values("CO2 emission (Tons)", ascending=False).head(20)